# Suite TSPRS — Parsing TypeScript

The standards of TypeScript and ECMAScript parse source text into TypeScript trees by delegating to
tree-sitter-typescript, then rewriting its concrete syntax tree into the kinds of typescript-estree. The converter
regroups what tree-sitter-typescript 0.23 groups unlike TypeScript, and a pre-pass handles the constructs it cannot
parse. Each case parses constructs and checks the tree (by shape where it matters, and by the text it prints), then
the errors, each with its line and column. In TypeScript, typescript-estree itself checks the conformance sources too.

In [ ]:
import { readFileSync } from "node:fs";

import { parse as estree } from "@typescript-eslint/typescript-estree";

import { Errors, Syntax as F } from "@mbse/programs/Framework";
import { ECMAScript2020, ECMAScript2025, Syntax as S, TypeScript50, TypeScript59 } from "@mbse/programs/TypeScript";
import * as Parser from "@mbse/programs/TypeScript/_Parser";
import { TypeScriptStandard } from "@mbse/programs/TypeScript/_Standard";

import { assert, equal, raises } from "./support.js";

const { ParseError } = Errors;
const TS = TypeScript59.STANDARD;
const TSX = TypeScript59.JSX;
const SOURCES = "../../conformance/sources";
type Standard = { parse(text: string): S.Program; print(node: F.Node): string; jsx: boolean };

function parse(text: string, standard: Standard = TS): S.Program {
  const program = standard.parse(text);
  equal(S.LANGUAGE.validate(program), []);
  return program;
}

const first = (text: string, standard: Standard = TS): any => parse(text, standard).body[0];
/** The expression of the expression statement `text`. */
const value = (text: string, standard: Standard = TS): any => first(text, standard).expression;

/** Parses `text`, checks it prints `expected` (by default itself) and that the print parses to the same tree. */
function sameText(text: string, expected: string | null = null, standard: Standard = TS): S.Program {
  const program = parse(text, standard);
  const printed = standard.print(program);
  assert(printed === (expected ?? text) + "\n", printed);
  assert(F.same(parse(printed, standard), program));
  return program;
}

function fails(text: string, message: string, line: number, column: number, standard: Standard = TS): void {
  raises(ParseError, () => standard.parse(text), message);
  try {
    standard.parse(text);
  } catch (error) {
    const e = error as InstanceType<typeof ParseError>;
    equal([e.line, e.column], [line, column]);
  }
}

const kinds = (nodes: readonly F.Node[]) => nodes.map((n) => n.kind().KIND);

// The trees of typescript-estree, and ours in their shape: TSESTree's own field names, without what each leaves out.
const RENAMED: Record<string, string> = {
  declarationKind: "kind", methodKind: "kind", propertyKind: "kind", moduleKind: "kind", isAsync: "async",
  isAwait: "await", isIn: "in", isOut: "out",
};
const MAPPED: Record<string, unknown> = { "?": true, "+?": "+", "-?": "-", readonly: true, "+readonly": "+",
  "-readonly": "-" };
const DROPPED = new Set(["loc", "range", "parent", "directive", "expression@ArrowFunctionExpression", "sourceType",
  "global", "assertions", "members@TSEnumDeclaration", "typeParameter@TSMappedType", "argument@TSImportType",
  "value@JSXText", "comments", "tokens", "value@Literal", "regex", "bigint", "generator@ArrowFunctionExpression",
  "id@ArrowFunctionExpression", "expression@FunctionExpression", "expression@FunctionDeclaration",
  "expression@TSDeclareFunction", "expression@TSEmptyBodyFunctionExpression", "declare@FunctionDeclaration",
  "declare@FunctionExpression", "declare@TSEmptyBodyFunctionExpression", "id@TSEmptyBodyFunctionExpression",
  "body@TSEmptyBodyFunctionExpression", "body@TSDeclareFunction", "abstract@ClassExpression", "declare@ClassExpression",
  "value@TemplateElement", "prefix@UnaryExpression", "phase@ImportExpression", "exportKind@ExportDefaultDeclaration"]);
const unset = (v: unknown) => v === null || v === undefined || v === false || (Array.isArray(v) && v.length === 0);

function theirs(node: any): any {
  if (Array.isArray(node)) return node.map((x) => (x === null ? { type: "Elision" } : theirs(x)));
  if (node === null || typeof node !== "object") return node;
  const out: any = {};
  for (const key of Object.keys(node).sort()) {
    if (DROPPED.has(key) || DROPPED.has(`${key}@${node.type}`) || unset(node[key])) continue;
    out[key] = theirs(node[key]);
  }
  if (node.type === "TemplateElement") out.raw = node.value.raw;
  if (node.type === "ImportExpression" && node.phase) out.phase = node.phase;
  return out;
}

function ours(node: any): any {
  if (node instanceof S.ParenthesizedExpression) return ours(node.expression);
  if (node instanceof S.TSParenthesizedType) return ours(node.typeAnnotation);
  const out: any = { type: node.kind().KIND };
  for (const field of node.kind().FIELDS) {
    let v = node.field(field.name);
    if (node instanceof S.Program && field.name === "hashbang") continue;
    if (Array.isArray(v)) v = v.filter((x) => !(x instanceof S.Comment)).map(ours);
    else if (v instanceof F.Node) v = ours(v);
    else if (node instanceof S.TSMappedType && (field.name === "optional" || field.name === "readonly") && v !== null) {
      v = MAPPED[v];
    }
    if (!unset(v)) out[RENAMED[field.name] ?? field.name] = v;
  }
  return Object.fromEntries(Object.entries(out).sort(([a], [b]) => (a < b ? -1 : a > b ? 1 : 0)));
}

## TSPRS-01 · The conformance sources parse to valid trees, which print and parse back the same, and hold every kind

In [ ]:
const seen = new Set<string>();
for (const [name, standard] of [["typescript59.ts", TS], ["typescript59-jsx.tsx", TSX]] as [string, Standard][]) {
  const text = readFileSync(`${SOURCES}/${name}`, "utf8");
  const program = parse(text, standard);
  const printed = standard.print(program);
  assert(F.same(parse(printed, standard), program) && standard.print(parse(printed, standard)) === printed);
  const [parsed, positions] = Parser.parse(text, standard.jsx);
  assert([...F.walk(parsed)].every((n) => positions.has(n))); // every node is placed, for the standard's problems
  // what typescript-estree reads is the same, of the source and of its print
  const reference = theirs(estree(text, { jsx: standard.jsx }));
  equal(ours(program), reference);
  equal(theirs(estree(printed, { jsx: standard.jsx })), reference);
  for (const n of F.walk(program)) seen.add(n.kind().KIND);
}
equal(seen, new Set(S.LANGUAGE.kinds().keys()));
console.log(seen.size, "kinds");

## TSPRS-02 · Literals keep their spelling: numbers, bigints, strings, regular expressions and templates

In [ ]:
for (const literal of ["0", "1_000", "0xFF", "0o17", "0b1010", "1.5e-3", ".5", "5.", "10n", "0xFFn", "'a'", "\"a\"",
  "'\\n\\u00e9'", "/a+/gi", "/[/]/v", "true", "false", "null"]) {
  assert(F.same(value(literal), new S.Literal({ raw: literal })), literal);
  sameText(`${literal};`);
}
assert(value("undefined") instanceof S.Identifier);
const template = value("`a${b}c${`d${e}`}`");
equal(template.quasis.map((q: any) => q.raw), ["a", "c", ""]);
equal(template.quasis.map((q: any) => q.tail), [false, false, true]);
assert(template.expressions[1] instanceof S.TemplateLiteral);
equal(value("`\\n${x}\\``").quasis.map((q: any) => q.raw), ["\\n", "\\`"]);
assert(value("String.raw`x`") instanceof S.TaggedTemplateExpression);
for (const text of ["`a${b}c`;", "String.raw`x${y}`;", "`\\n${x}\\``;"]) sameText(text);
console.log("ok");

## TSPRS-03 · Expressions, with TypeScript's precedence where tree-sitter-typescript's differs

In [ ]:
let grouped = value("a ?? b as T"); // `as` binds tighter than `??`
assert(grouped instanceof S.LogicalExpression && grouped.right instanceof S.TSAsExpression);
grouped = value("a || b && c as number < d");
assert(grouped.operator === "||" && grouped.right.operator === "&&" && grouped.right.right.operator === "<");
assert(grouped.right.right.left instanceof S.TSAsExpression);
grouped = value("a ? b : c as T");
assert(grouped instanceof S.ConditionalExpression && grouped.alternate instanceof S.TSAsExpression);
assert(value("x = a satisfies T").right.typeAnnotation !== null);
assert(value("f = () => a as T").right.body instanceof S.TSAsExpression);
assert(value("a == b as T == c").left.operator === "==");
const member = value("x as A.B.C"); // the names after `as` are the type's
assert(member instanceof S.TSAsExpression);
equal(S.LANGUAGE.validate(member), []);
assert(TS.print(member.typeAnnotation) === "A.B.C");
assert(TS.print(value("x as A | B.C.D").typeAnnotation) === "A | B.C.D");
assert(TS.print(value("x satisfies A & B.C.D").typeAnnotation) === "A & B.C.D");
assert(value("(x as T).y") instanceof S.MemberExpression);
assert(value("x as T[]") instanceof S.TSAsExpression);
assert(value("x as T[].y").object instanceof S.TSAsExpression); // only names continue a type
assert(value("[1] as const").typeAnnotation.typeName.name === "const");
assert(value("a ?? b as const").right instanceof S.TSAsExpression); // `as const` binds as `as` does
assert(value("a + b as T") instanceof S.TSAsExpression); // `+` binds tighter than `as`
const asserted = value("a ?? b!");
assert(asserted instanceof S.LogicalExpression && asserted.right instanceof S.TSNonNullExpression);
assert(value("a ? b : c!").alternate instanceof S.TSNonNullExpression);
assert(value("x = y!").right instanceof S.TSNonNullExpression);
const chain = value("a?.b.c!.d?.[e]?.(f)");
assert(chain instanceof S.ChainExpression && chain.expression.optional);
assert(value("a!.b") instanceof S.MemberExpression && !(value("a.b") instanceof S.ChainExpression));
assert(value("(a?.b).c").object instanceof S.ParenthesizedExpression);
equal(kinds(value("a, b, c").expressions), ["Identifier", "Identifier", "Identifier"]);
assert(value("x++").prefix === false && value("--x").prefix === true);
assert(value("f<T>(a)").typeArguments !== null && value("f<T>") instanceof S.TSInstantiationExpression);
assert(value("new A<T>") instanceof S.NewExpression);
equal(value("new A").arguments, []);
assert(value("import('m', { with: {} })") instanceof S.ImportExpression && value("import('m')").options === null);
assert(value("<T>x") instanceof S.TSTypeAssertion);
assert(value("#x in o").left instanceof S.PrivateIdentifier);
assert(value("a **= b").operator === "**=" && value("a >>>= b").operator === ">>>=");
assert(value("yield* g").delegate === true && value("yield").argument === null);
assert(value("async (a) => { await a; }").isAsync === true && value("a => a").params[0] instanceof S.Identifier);
equal(kinds(value("[a, , ...b]").elements), ["Identifier", "Elision", "SpreadElement"]);
const literal = value("({ a, b: 1, [c]: 2, 'd': 3, 4: 5, m() {}, get g() { return 1; }, set s(v) {}, async *k() {}, ...r })");
const properties = literal.expression.properties;
equal(properties.slice(0, 8).map((p: any) => p.propertyKind), ["init", "init", "init", "init", "init", "init", "get", "set"]);
assert(properties[0].shorthand && properties[2].computed && properties[5].method);
assert(properties.at(-1) instanceof S.SpreadElement);
const meta = value("new.target");
equal([meta.meta.name, meta.property.name], ["new", "target"]);
assert(value("import.meta").meta.name === "import");
for (const text of ["a ?? b as T;", "a || b && c as number < d;", "a?.b.c!.d?.[e]?.(f);", "x = (a, b);", "f<T>(a);",
  "new A<T>();", "new.target;", "(x as T).y;", "x as A.B.C;", "<T>x;", "#x in o;", "a **= b;",
  "async (a) => {\n    await a;\n};", "(a) => a;", "a?.(b)!;", "(await a) ** 2;", "(-a) ** 2;"]) {
  sameText(text);
}
sameText("new A;", "new A();");
console.log("ok");

## TSPRS-04 · Statements: declarations of variables, conditions, loops, switches, exceptions, labels and jumps

In [ ]:
const declaration = first("let a = 1, [b, , ...c] = d, { e, f: g = 2, ...h } = i;");
assert(declaration.declarationKind === "let" && declaration.declarations.length === 3);
const pattern = declaration.declarations[2].id;
equal(kinds(pattern.properties), ["Property", "Property", "RestElement"]);
assert(pattern.properties[0].shorthand && pattern.properties[1].value instanceof S.AssignmentPattern);
const defaulted = first("const { a = 1 } = b;").declarations[0].id.properties[0];
assert(defaulted.shorthand && defaulted.value instanceof S.AssignmentPattern);
assert(first("let x!: number;").declarations[0].definite === true);
assert(first("var x;").declarations[0].init === null);
assert(first("using r = f();").declarationKind === "using");
assert(first("await using r = f();").declarationKind === "await using");
assert(first("for (using x of y) {}").left.declarationKind === "using");
assert(first("for (await using x of y) {}").left.declarationKind === "await using");
assert(first("for await (const x of y) {}").isAwait === true);
assert(first("for (var x = 1 in o) {}").left.declarations[0].init.raw === "1");
assert(first("for (x in y) z;").left instanceof S.Identifier);
assert(first("for (;;) {}").init === null && first("for (i = 0; i < 1; i++) {}").test !== null);
assert(first("for (let i = 0;;) {}").init instanceof S.VariableDeclaration);
const switched = first("switch (a) { case 1: b; break; default: c; }");
equal(switched.cases.map((c: any) => c.test === null), [false, true]);
assert(switched.cases[0].consequent.length === 2);
equal(first("switch (a) { // dropped: before every case\n}").cases, []);
const attempt = first("try { a; } catch (e: unknown) { b; } finally { c; }");
assert(attempt.handler.param.typeAnnotation !== null && attempt.finalizer !== null);
assert(first("try {} catch {}").handler.param === null && first("try {} finally {}").handler === null);
assert(first("try {} catch ({ message }) {}").handler.param.properties[0].shorthand);
assert(first("if (a) b; else if (c) d; else e;").alternate.alternate !== null);
assert(first("l: for (;;) { break l; continue l; }").body.body.body[0].label.name === "l");
assert(first("throw e;").argument.name === "e");
assert((parse("function f() { return; }").body[0] as any).body.body[0] instanceof S.ReturnStatement);
sameText("function f() {\n    return;\n}");
for (const text of ["let x!: number;", "for (using x of y) {}", "for (await using x of y) {}", "do {} while (a);",
  "do\n    a;\nwhile (b);", "while (a)\n    b;", "with (o) {}", "debugger;", ";", "if (a) {} else {}",
  "switch (a) {\n    case 1:\n        b;\n        break;\n    default:\n        c;\n}",
  "try {} catch (e: unknown) {} finally {}", "l: for (;;) {\n    break l;\n    continue l;\n}",
  "for (x in y)\n    z;", "for (const [a, b] of c) {}"]) {
  sameText(text);
}
console.log("ok");

## TSPRS-05 · Functions and classes: signatures, overloads, members, modifiers, decorators and parameter properties

In [ ]:
const fn = first("export async function* f<T>(this: X, a?: T, b = 1, ...c: T[]): AsyncGenerator<T> {}").declaration;
assert(fn.isAsync && fn.generator);
equal(kinds(fn.params), ["Identifier", "Identifier", "AssignmentPattern", "RestElement"]);
assert(fn.params[1].optional && fn.returnType !== null);
assert(first("declare function f(): void;") instanceof S.TSDeclareFunction);
assert(first("declare function f(): void;").declare === true);
assert(first("function f(a: string): string;") instanceof S.TSDeclareFunction);
assert(first("function g(x: unknown): x is string { return true; }").returnType.typeAnnotation.parameterName.name === "x");
assert(first("function g(x: unknown): asserts x {}").returnType.typeAnnotation.asserts === true);
assert(first("function g(this: X): asserts this is Y {}").returnType.typeAnnotation.typeAnnotation !== null);
const klass = first(`@d @e.f()
abstract class C<T> extends B<T> implements I, J<T> {
    #p = 1;
    static q?: number;
    declare readonly r: string;
    protected override s!: number;
    accessor t = 1;
    abstract u: string;
    abstract accessor w: number;
    [k: string]: unknown;
    constructor(private readonly a: T, public b = 1, @g c?: number) { super(); }
    get v(): number { return 1; }
    set v(x) {}
    static { C.q = 1; }
    m?(): void;
    abstract n(): void;
    'constructor'() {}
    [Symbol.iterator]() {}
    @h async *o() {}
}`);
assert(klass.abstract && klass.decorators.length === 2 && klass.implements.length === 2);
assert(klass.superTypeArguments !== null);
const members = klass.body.body;
equal(kinds(members), [
  "PropertyDefinition", "PropertyDefinition", "PropertyDefinition", "PropertyDefinition", "AccessorProperty",
  "TSAbstractPropertyDefinition", "TSAbstractAccessorProperty", "TSIndexSignature", "MethodDefinition",
  "MethodDefinition", "MethodDefinition", "StaticBlock", "MethodDefinition", "TSAbstractMethodDefinition",
  "MethodDefinition", "MethodDefinition", "MethodDefinition"]);
assert(members[0].key instanceof S.PrivateIdentifier && members[1].static && members[1].optional);
assert(members[2].declare && members[2].readonly && members[3].accessibility === "protected" && members[3].override);
assert(members[3].definite);
const constructor = members[8];
assert(constructor.methodKind === "constructor");
equal(kinds(constructor.value.params), ["TSParameterProperty", "TSParameterProperty", "Identifier"]);
assert(constructor.value.params[0].readonly && constructor.value.params[1].accessibility === "public");
assert(constructor.value.params[2].decorators.length === 1);
equal(members.slice(9, 11).map((m: any) => m.methodKind), ["get", "set"]);
assert(members[12].optional && members[12].value instanceof S.TSEmptyBodyFunctionExpression);
assert(members[14].methodKind === "constructor" && members[15].computed);
assert(members[16].decorators.length === 1 && members[16].value.isAsync && members[16].value.generator);
assert(first("class D { static constructor() {} }").body.body[0].methodKind === "method");
const heritage = first("class X extends B implements a.b.C, d.E<T> {}");
assert(heritage.superTypeArguments === null && TS.print(heritage.implements[0]) === "a.b.C");
assert(first("interface I extends a.B {}").extends[0].expression instanceof S.MemberExpression);
assert(value("(class { })") instanceof S.ParenthesizedExpression);
assert(first("export default class {}").declaration instanceof S.ClassDeclaration);
assert(first("export default function () {}").declaration.id === null);
assert(first("export default function* g() {}").declaration instanceof S.FunctionDeclaration);
assert(first("@d export class E {}").declaration instanceof S.ClassDeclaration);
assert(first("@d export class E {}").declaration.decorators.length === 1);
assert(first("@d export default class {}").declaration.decorators.length === 1);
for (const text of ["declare function f(): void;", "function f(a: string): string;",
  "class C {\n    constructor(private readonly a: T, public b = 1) {}\n}",
  "class C {\n    #p = 1;\n    static q?: number;\n    accessor t = 1;\n}", "@d\nexport class E {}",
  "export default class {}", "export default function() {}",
  "class C {\n    get v(): number {\n        return 1;\n    }\n    set v(x) {}\n}",
  "abstract class A {\n    abstract n(): void;\n}", "class C {\n    async *o() {}\n}"]) {
  sameText(text);
}
console.log("ok");

## TSPRS-06 · Modules: imports, exports, attributes, import equals, namespaces, ambient modules and `declare global`

In [ ]:
const imported = first("import d, { a as b, type c, 'e' as f } from 'm' with { type: 'json' };");
equal(kinds(imported.specifiers), ["ImportDefaultSpecifier", "ImportSpecifier", "ImportSpecifier", "ImportSpecifier"]);
assert(imported.specifiers[2].importKind === "type" && imported.specifiers[3].imported instanceof S.Literal);
assert(imported.attributes[0].key.name === "type");
assert(first("import type { T } from 'm';").importKind === "type");
assert(first("import * as ns from 'm';").specifiers[0].local.name === "ns");
equal(first("import 'side';").specifiers, []);
assert(first("import x = require('m');").moduleReference.expression.raw === "'m'");
assert(first("import type x = require('m');").importKind === "type");
assert(first("import y = A.B.C;").moduleReference instanceof S.TSQualifiedName);
assert(first("export * as ns from 'm';").exported.name === "ns");
assert(first("import a from 'm' with { 'type': 'json' };").attributes[0].key instanceof S.Literal);
assert(first("export * as 'n s' from 'm';").exported instanceof S.Literal);
const named = first("export { a as b, type c, 'd' } from 'm';");
equal(named.specifiers.map((s: any) => s.exportKind), ["value", "type", "value"]);
assert(named.source !== null && named.specifiers[2].exported instanceof S.Literal);
assert(first("export { a as 'b c' };").specifiers[0].exported.raw === "'b c'");
assert(first("export type { T };").exportKind === "type");
assert(first("export interface I {}").exportKind === "type");
assert(first("export declare const x: number;").exportKind === "type");
assert(first("export const y = 1;").exportKind === "value");
assert(first("export = x;") instanceof S.TSExportAssignment);
assert(first("export as namespace N;").id.name === "N");
assert(first("export default a + b;").declaration instanceof S.BinaryExpression);
const namespace = first("namespace A.B { export const x = 1; }");
assert(namespace.id instanceof S.TSQualifiedName && namespace.moduleKind === "namespace");
assert(first("module M {}").moduleKind === "module");
const ambient = parse("declare module \"m\" { export const v: number; }\ndeclare module \"short\";\nx;").body as any[];
assert(ambient[0].declare && ambient[0].id instanceof S.Literal && ambient[1].body === null);
equal(kinds(ambient), ["TSModuleDeclaration", "TSModuleDeclaration", "ExpressionStatement"]);
assert(first("declare global { interface W {} }").moduleKind === "global");
const inModule = first("declare module \"m\" { global { interface W {} } }").body.body[0];
assert(inModule.moduleKind === "global" && inModule.declare === false);
assert(first("type A<T> = T[];").typeParameters.params[0].name.name === "T");
assert(first("type U<S extends string> = intrinsic;").typeAnnotation instanceof S.TSIntrinsicKeyword);
const iface = first("interface I<T> extends A.B<T>, C { a: T; }");
assert(iface.extends[0].expression instanceof S.MemberExpression && iface.extends[0].typeArguments !== null);
const enumerated = first("declare const enum E { A, B = A + 1, 'C-d', [k] = 2 }");
assert(enumerated.const && enumerated.declare);
equal(enumerated.body.members.map((m: any) => m.computed), [false, false, false, true]);
for (const text of ["import d, {a as b, type c} from 'm' with { type: 'json' };", "import type {T} from 'm';",
  "import * as ns from 'm';", "import 'side';", "import x = require('m');", "import y = A.B.C;",
  "export * as ns from 'm';", "export * from 'm';", "export {a as b, type c} from 'm';", "export type {T};",
  "export = x;", "export as namespace N;", "export default a + b;", "namespace A.B {\n    export const x = 1;\n}",
  "declare module \"short\";", "declare global {\n    interface W {}\n}", "type A<T> = T[];",
  "declare const enum E {\n    A,\n    B = A + 1,\n}", "export default (function() {});", "export default (a, b);"]) {
  sameText(text);
}
console.log("ok");

## TSPRS-07 · Types, with TypeScript's grouping where tree-sitter-typescript's differs

In [ ]:
const typeOf = (text: string): any => first(`type X = ${text};`).typeAnnotation;

equal(kinds(typeOf("any | unknown | number | bigint | boolean | string | symbol | object | never | void | undefined | "
  + "null | this").types), [
  "TSAnyKeyword", "TSUnknownKeyword", "TSNumberKeyword", "TSBigIntKeyword", "TSBooleanKeyword", "TSStringKeyword",
  "TSSymbolKeyword", "TSObjectKeyword", "TSNeverKeyword", "TSVoidKeyword", "TSUndefinedKeyword", "TSNullKeyword",
  "TSThisType"]);
assert(typeOf("unique symbol").operator === "unique");
const readonly = typeOf("readonly A[] | B"); // `readonly` binds tighter than `|`
assert(readonly instanceof S.TSUnionType && readonly.types[0].operator === "readonly");
equal(kinds(typeOf("readonly A[] | B | C").types), ["TSTypeOperator", "TSTypeReference", "TSTypeReference"]);
assert(typeOf("readonly A[] & B") instanceof S.TSIntersectionType);
equal(kinds(typeOf("A | readonly B[] | C").types), ["TSTypeReference", "TSTypeOperator", "TSTypeReference"]);
assert(typeOf("keyof T").operator === "keyof");
equal(typeOf("`a${B}c`").quasis.map((q: any) => q.raw), ["a", "c"]);
assert(typeOf("-1").literal instanceof S.UnaryExpression && typeOf("'a'").literal.raw === "'a'");
const tupleType = typeOf("[a: string, b?: number, ...c: boolean[]]").elementTypes;
equal(kinds(tupleType), ["TSNamedTupleMember", "TSNamedTupleMember", "TSRestType"]);
assert(tupleType[1].optional);
equal(kinds(typeOf("[string?, ...number[]]").elementTypes), ["TSOptionalType", "TSRestType"]);
assert(typeOf("abstract new () => object") instanceof S.TSConstructorType);
assert(typeOf("<T>(a: T) => a is string").returnType.typeAnnotation.parameterName.name === "a");
assert(typeOf("(a: unknown) => asserts a").returnType.typeAnnotation.asserts);
const mapped = typeOf("{ -readonly [K in keyof T as `get${K & string}`]+?: T[K] }");
equal([mapped.readonly, mapped.optional], ["-readonly", "+?"]);
assert(mapped.nameType !== null);
equal([typeOf("{ readonly [K in T]-?: K }").readonly, typeOf("{ [K in T]?: K }").optional], ["readonly", "?"]);
assert(typeOf("{ +readonly [K in T]: K }").readonly === "+readonly");
const conditional = typeOf("T extends (infer U extends string)[] ? U : never");
assert(conditional.extendsType.elementType.typeAnnotation.typeParameter.constraint !== null);
assert(typeOf("typeof a.b<T>").typeArguments !== null);
assert(typeOf("typeof import('m')") instanceof S.TSTypeQuery);
const importType = typeOf("import('m', { with: { 'resolution-mode': 'import' } }).A.B<T>");
assert(importType instanceof S.TSImportType && importType.options !== null && importType.typeArguments !== null);
assert(TS.print(importType.qualifier) === "A.B");
assert(typeOf("import('m')") instanceof S.TSImportType && typeOf("import('m').X").qualifier.name === "X");
assert(first("let a: typeof import('m').x;").declarations[0].id.typeAnnotation.typeAnnotation.exprName instanceof
  S.TSImportType);
assert(typeOf("A['b'][C]") instanceof S.TSIndexedAccessType);
const typeLiteral = typeOf("{ a: string; readonly b?: number; c(): void; get d(): number; set d(v); (x): y; new (x): y; "
  + "[k: string]: unknown; static e: string }");
equal(kinds(typeLiteral.members), [
  "TSPropertySignature", "TSPropertySignature", "TSMethodSignature", "TSMethodSignature", "TSMethodSignature",
  "TSCallSignatureDeclaration", "TSConstructSignatureDeclaration", "TSIndexSignature", "TSPropertySignature"]);
assert(typeLiteral.members[1].readonly && typeLiteral.members[3].methodKind === "get");
assert(typeOf("(A)") instanceof S.TSParenthesizedType && typeOf("A.B.C").typeName instanceof S.TSQualifiedName);
assert(typeOf("bigint") instanceof S.TSBigIntKeyword);
assert(typeOf("typeof this").exprName instanceof S.ThisExpression);
assert(typeOf("new <T>() => T").typeParameters !== null);
equal(typeOf("{ a; b? }").members.map((m: any) => m.typeAnnotation), [null, null]);
assert(typeOf("T extends (infer U)[] ? U : never").extendsType.elementType.typeAnnotation.typeParameter.constraint === null);
assert(typeOf("readonly string[]") instanceof S.TSTypeOperator);
for (const text of ["type X = readonly A[] | B;", "type X = keyof T;", "type X = [a: string, b?: number, ...c: boolean[]];",
  "type X = abstract new () => object;", "type X = { -readonly [K in keyof T as `get${K & string}`]+?: T[K] };",
  "type X = T extends (infer U extends string)[] ? U : never;", "type X = typeof a.b<T>;",
  "type X = import('m').A.B<T>;", "type X = A['b'][C];", "type X = (A | B)[];", "type X = unique symbol;",
  "type X = `a${B}c`;", "type X = -1;", "type X = { a: string; readonly b?: number; c(): void; };",
  "type X = (() => void) | null;", "type X = <T>(a: T) => a is string;", "type X = typeof this;",
  "type X = new <T>() => T;", "type X = readonly string[];", "class X extends B implements a.b.C {}"]) {
  sameText(text);
}
console.log("ok");

## TSPRS-08 · JSX, with the `tsx` grammar: elements, fragments, attributes, children and the text between them

In [ ]:
const element = value("<a.b x='1' y={2} {...z} c:d=\"e\" f>t &amp; u{v}{/* c */}{...w}<g /><>h</></a.b>", TSX);
assert(element.openingElement.name instanceof S.JSXMemberExpression);
equal(kinds(element.openingElement.attributes), [
  "JSXAttribute", "JSXAttribute", "JSXSpreadAttribute", "JSXAttribute", "JSXAttribute"]);
assert(element.openingElement.attributes[3].name instanceof S.JSXNamespacedName);
assert(element.openingElement.attributes[4].value === null);
equal(kinds(element.children), ["JSXText", "JSXExpressionContainer", "JSXExpressionContainer", "JSXSpreadChild",
  "JSXElement", "JSXFragment"]);
assert(element.children[0].raw === "t &amp; u" && element.children[2].expression instanceof S.JSXEmptyExpression);
assert(value("<C<T> />", TSX).openingElement.typeArguments !== null);
assert(value("<div>\n    a\n</div>", TSX).children[0].raw === "\n    a\n");
assert(value("<div>  </div>", TSX).children[0].raw === "  ");
equal(value("<div></div>", TSX).children, []);
assert(value("<T,>(a: T) => a", TSX).typeParameters !== null);
for (const text of ["<a.b x='1' y={2} {...z} f>t &amp; u{v}{...w}<g /><>h</></a.b>;", "<C<T> />;", "<div>{}</div>;",
  "const f = <T,>(a: T) => a;"]) {
  sameText(text, null, TSX);
}
console.log("ok");

## TSPRS-09 · Comments are kept where statements and members are listed, on their own lines or trailing

In [ ]:
const commented = parse(`#!/usr/bin/env node
// one
a; /* two */
class C { // three
    // four
    m() {} // five
}
interface I { /* six */ a: string; }
enum E { A, // seven
}
switch (x) { case 1: // eight
}
f(/* dropped */ a);
`);
assert(commented.hashbang === "/usr/bin/env node");
const listed = commented.body as any[];
equal(kinds(listed), ["Comment", "ExpressionStatement", "Comment", "ClassDeclaration", "TSInterfaceDeclaration",
  "TSEnumDeclaration", "SwitchStatement", "ExpressionStatement"]);
equal([listed[0].trailing, listed[2].trailing, listed[2].block], [false, true, true]);
equal(listed[3].body.body.filter((c: any) => c instanceof S.Comment).map((c: any) => [c.text, c.trailing]), [
  [" three", true], [" four", false], [" five", true]]);
assert(listed[4].body.body[0].text === " six " && listed[5].body.members[1].trailing);
assert(listed[6].cases[0].consequent[0].trailing);
sameText(`a; /* two */
class C { // three
    // four
    m() {} // five
}
enum E {
    A, // seven
}
type T = {
    // eight
    a: string;
};`);
console.log("ok");

## TSPRS-10 · What tree-sitter-typescript 0.23 cannot parse, which the pre-pass puts back

In [ ]:
const accessors = first("class C { accessor a = 1; static accessor b: number; accessor() {} accessor = 2; accessor }");
equal(kinds(accessors.body.body), ["AccessorProperty", "AccessorProperty", "MethodDefinition", "PropertyDefinition",
  "PropertyDefinition"]);
equal(accessors.body.body.map((m: any) => m.key.name), ["a", "b", "accessor", "accessor", "accessor"]);
assert(accessors.body.body[1].static);
const parameters = first("type T<in A, out B, in out C, const in D> = A;").typeParameters.params;
equal(parameters.map((p: any) => [p.isIn, p.isOut]), [[true, false], [false, true], [true, true], [true, false]]);
assert(parameters[3].const && first("function f<in>() {}").typeParameters.params[0].name.name === "in");
assert(first("import defer * as m from 'm';").phase === "defer" && first("import { defer } from 'm';").phase === null);
assert(first("import defer { a } from 'm';").phase === "defer");
assert(first("export type * from 'm';").exportKind === "type");
assert(first("export type * as ns from 'm';").exportKind === "type");
const abstract = first("class C { abstract: number; readonly abstract?: string; abstract() {} }").body.body;
equal(abstract.map((m: any) => m.key.name), ["abstract", "abstract", "abstract"]);
assert(value("using + 1").left.name === "using" && value("x = using").right.name === "using");
assert(value("using in o").left.name === "using");
assert(first("let x: import('m').A<B>;").declarations[0].id.typeAnnotation.typeAnnotation.typeArguments !== null);
const spanning = parse("let x: import(\n'm').A<B>;\n// after\ny;").body as any[]; // the import spans lines
assert(spanning[0].declarations[0].id.typeAnnotation.typeAnnotation.typeArguments !== null);
assert(!spanning[1].trailing); // lines are those written
assert(value("a<import('m').B>()").typeArguments.params[0] instanceof S.TSImportType);
for (const text of ["class C {\n    accessor a = 1;\n    static accessor b: number;\n}", "type T<in A, out B, in out C> = A;",
  "import defer * as m from 'm';", "export type * from 'm';", "class C {\n    abstract: number;\n}", "using + 1;",
  "let x: import('m').A<B>;", "a<import('m').B>();"]) {
  sameText(text);
}
console.log("ok");

## TSPRS-11 · JavaScript: the editions of ECMAScript parse what has no TypeScript syntax, and reject the rest

In [ ]:
const ES = ECMAScript2025.STANDARD;
const javascript = parse("const { a = 1 } = b;\nclass C { #x; static { } }\nlabel: for (const y of z) break label;", ES);
equal(kinds(javascript.body), ["VariableDeclaration", "ClassDeclaration", "LabeledStatement"]);
assert(parse("<div />;", ECMAScript2025.JSX).body[0].expression instanceof S.JSXElement);
fails("let a: number;", "TSTypeAnnotation is not ECMAScript, but this is ES2025", 1, 6, ES);
fails("enum E {}", "TSEnumDeclaration is not ECMAScript, but this is ES2025", 1, 1, ES);
fails("@d class C {}", "Decorator is not ECMAScript, but this is ES2025", 1, 1, ES);
fails("a ??= b;", "AssignmentExpression.operator '??=' needs ES2021, but this is ES2020", 1, 1, ECMAScript2020.STANDARD);
console.log("ok");

## TSPRS-12 · Errors name their line and column, counted in characters

In [ ]:
fails("let = ;", "syntax error", 1, 5);
fails("a\n😀; b +", "syntax error", 2, 4); // columns count characters, not UTF-16 units
fails("if (a {}", "expected )", 1, 6);
fails("import defer * as m from 'm';", "ImportDeclaration.phase 'defer' needs TypeScript 5.9, but this is TypeScript 5.0",
  1, 1, TypeScript50.STANDARD);
fails("<div />;", "syntax error", 1, 1);
fails("<T>x;", "syntax error", 1, 1, TSX);
fails("let s = 'é😀'; x satisfies T;", "TSSatisfiesExpression needs TypeScript 4.9, but this is TypeScript 4.8", 1, 15,
  new TypeScriptStandard(4, 8));
fails("type X = { [K in T]: K; a: string };", "unsupported syntax: index_signature", 1, 12);
fails("x = (a: T);", "unsupported syntax: parenthesized_expression", 1, 5);
fails("type X = ?string;", "unsupported syntax: flow_maybe_type", 1, 10); // Flow's
fails("let x: import('m'", "syntax error", 1, 1);
console.log("ok");